# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [8]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [9]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [12]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [15]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [17]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [18]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [19]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'forum page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [20]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [21]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
6 days ago
•
627k
•
913
openbmb/MiniCPM-V-4.6
Updated
about 18 hours ago
•
16.8k
•
529
HiDream-ai/HiDream-O1-Image
Updated
2 days ago
•
9.86k
•
324
Zyphra/ZAYA1-8B
Updated
3 days ago
•
131k
•
493
deepseek-ai/DeepSeek-V4-Pro
Updated
9 days ago
•
2.59M
•
3.95k
Browse 2M+ models
Spaces
Paused
MCP
1.16k
Wan2.2 14B Fast Preview
🐌
1.16k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured
94
Pixal3D
🏆
94
High-fidelity pixel-aligned image-to-3D generation.
Running
156
The ultim

In [24]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\n6 days ago\n•\n627k\n•\n913\nopenbmb/MiniCPM-V-4.6\nUpdated\nabout 18 hours ago\n•\n16.8k\n•\n529\nHiDream-ai/HiDream-O1-Image\nUpdated\n2 days ago\n•\n9.86k\n•\n324\nZyphra/ZAYA1-8B\nUpdated\n3 days ago\n•\n131k\n•\n493\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n9 days ago\n•\n2.59M\n•\n3.95k\nBrowse 2M+ models\nSpaces\nPaused\nMCP\n1.16k\nWan2.2 14B Fast 

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. Serving as a collaboration platform, Hugging Face empowers machine learning engineers, scientists, and AI users worldwide to create, share, explore, and innovate on state-of-the-art machine learning models, datasets, and applications.

At its core is the **Hugging Face Hub**, a central space hosting **2 million+ models**, **500,000+ datasets**, and **1 million+ applications**, enabling fast discovery and experimentation with open-source machine learning tools. Hugging Face is driving the open and ethical AI future by supporting collaboration and transparency.

---

## What We Offer

- **Hugging Face Hub**: Share and host unlimited public models, datasets, and apps. An essential resource for broad ML community engagement.
- **Multi-modality Support**: Work with text, image, video, audio, and even 3D data through our diverse libraries and tools.
- **Spaces**: Deploy and run customizable ML applications directly on the platform for sharing interactive demos and prototypes.
- **Enterprise Plans**: Tailored services designed to support businesses with advanced ML model hosting, collaboration, and scaling needs.
- **Open Source Stack**: Leverage the Hugging Face open-source ecosystem for faster and more efficient model development.
- **Cutting-edge Research**: A talented scientific team constantly pushing the boundaries of AI technology.

---

## Company Culture

Hugging Face is more than a company—it’s a vibrant, fast-growing community united by the mission to build an open, collaborative, and ethical AI ecosystem. The culture embraces transparency, openness, and innovation, providing a platform for anyone—from beginners to experts—to learn, contribute, and advance the AI revolution together.

The community-driven approach encourages sharing knowledge, improving accessibility, and fostering inclusivity in AI development worldwide.

---

## Our Customers and Partners

Hugging Face serves a diverse array of users:

- Individual ML engineers and researchers advancing AI technologies.
- Enterprises integrating powerful AI solutions for scalable business use.
- Academic institutions leveraging models and datasets for teaching and research.
- Developers building AI applications with ease and speed on Spaces.
- Organizations committed to ethical, transparent AI collaboration.

---

## Careers at Hugging Face

Join the AI revolution with Hugging Face! We seek passionate AI engineers, researchers, community builders, and software developers excited to contribute to open-source AI and help shape the future.

Working at Hugging Face means:

- Collaborating with top talent in AI and ML.
- Being part of a mission-driven culture focused on open and ethical AI.
- Access to cutting-edge research and technical resources.
- Opportunities for professional growth within a global, inclusive community.

Explore current openings on our [Careers page](https://huggingface.co/careers) and become part of the future of machine learning.

---

## Connect With Us

- **Website:** https://huggingface.co  
- **GitHub:** https://github.com/huggingface  
- **Twitter:** https://twitter.com/huggingface  
- **LinkedIn:** https://linkedin.com/company/hugging-face  
- **Discord:** Join the community for discussions, support, and collaboration.

---

*Hugging Face — Where the machine learning community builds tomorrow’s AI today.*  

#AI #MachineLearning #OpenSource #Collaboration #EthicalAI

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a leading AI community dedicated to building the future of machine learning. The platform serves as a collaborative hub where data scientists, developers, and researchers work together on machine learning models, datasets, and applications. With over 2 million models and half a million datasets available, Hugging Face empowers the global AI community to explore and innovate across all modalities including text, image, video, audio, and 3D data.

---

## What We Offer

- **Collaborative Platform:** Host and share unlimited public models, datasets, and applications.
- **Open Source Stack:** Leverage Hugging Face’s open-source tools for faster, more efficient machine learning development.
- **Model & Dataset Repository:** Access a vast library of models and datasets that are regularly updated by a vibrant, engaged community.
- **Spaces:** Explore and deploy interactive ML applications with ease.
- **Multimodal Support:** Work flexibly across data types - from natural language and images to videos and 3D models.
- **Build Your Portfolio:** Share your work with the AI community and establish your machine learning profile.

---

## Our Customers

Hugging Face’s platform is used by a diverse array of professionals and organizations including:

- AI Researchers and Engineers
- Enterprises and AI Teams seeking scalable ML solutions
- Developers building AI-powered applications
- Academic Institutions and Students enhancing their AI projects
- Enthusiasts and hobbyists contributing to open source AI innovation

---

## Enterprise Solutions

For businesses eager to integrate AI at scale, Hugging Face offers tailored team and enterprise plans. These come with advanced security, collaboration tools, and dedicated support to help AI teams move faster and more efficiently. Enterprise clients benefit from:

- Team collaboration at scale with instant setup
- Enhanced storage capacities and inference credits
- Priority compute resources including ZeroGPU quotas
- Support for deploying private models and datasets

---

## Pricing Plans

- **PRO Account:** For individuals seeking enhanced capabilities ($9/month), including increased storage, inference credits, prioritized compute, and personal blog publishing.
- **Team Plan:** Designed for growing teams ($20 per user/month), offering collaboration features and scaled resources.
- (Enterprise-level custom plans available.)

---

## Company Culture

Hugging Face fosters an open, inclusive, and community-driven culture. The company thrives on collaboration and transparency, encouraging contributors worldwide to share knowledge and advances in AI. Innovation is driven by mutual support, open-source contributions, and a shared passion for pushing the boundaries of machine learning technology.

---

## Careers

Join Hugging Face if you are passionate about AI and want to contribute to cutting-edge technologies that shape the future. The company values talent across:

- Machine Learning Research
- Software and AI Engineering
- Product Development
- Community & Developer Relations
- Enterprise Solutions and Customer Success

Hugging Face offers an engaging, mission-driven work environment with opportunities to learn, grow, and impact the AI landscape on a global scale.

---

## Connect With Us

Explore the future of AI with Hugging Face by visiting [huggingface.co](https://huggingface.co).  
Sign up to join the community, contribute models, datasets, or deploy your own AI applications today!

---

*Hugging Face – The AI community building the future.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>